In [0]:
# Celda 1: variables de la tabla que NO están en el diccionario
import pandas as pd
from pyspark.sql import functions as F

dic = pd.read_csv("/Volumes/workspace/default/raw/LCDataDictionary.csv")
dic_vars = set(dic["LoanStatNew"].dropna().str.strip())   # strip: hay nombres con espacios al final

cols_raw = spark.table("workspace.default.lc_accepted_raw").columns
fuera = sorted(set(cols_raw) - dic_vars)

print(f"Variables en la tabla: {len(cols_raw)} | en el diccionario: {len(dic_vars)} | sin documentar: {len(fuera)}")
for c in fuera:
    print(c)


In [0]:
# Celda 2: % de faltantes por variable y cosecha (sobre la población final)
pob = spark.table("workspace.default.lc_population")
cols = [c for c in pob.columns if c != "issue_year"]

# Una sola pasada sobre los datos: para cada variable, el promedio de "es nulo" = % de faltantes
faltantes = (pob.groupBy("issue_year")
                .agg(*[F.round(F.avg(F.col(c).isNull().cast("int")) * 100, 1).alias(c) for c in cols]))

# Transponer: filas = variables, columnas = cosechas (como un PROC TRANSPOSE)
pdf = faltantes.toPandas().set_index("issue_year").sort_index().T
pdf.columns = [str(c) for c in pdf.columns]
pdf["max_dev"] = pdf[["2013", "2014"]].max(axis=1)   # peor cosecha de desarrollo
pdf.index.name = "variable"

display(pdf.reset_index().sort_values("max_dev", ascending=False))


In [0]:
from pyspark.sql import functions as F

pob = spark.table("workspace.default.lc_population").filter(F.col("issue_year") >= 2013)

# ¿Los faltantes de utilización coinciden con clientes sin tarjetas bancarias?
display(pob.groupBy(F.col("bc_util").isNull().alias("bc_util_nulo"),
                    (F.col("num_bc_tl") == 0).alias("sin_tarjetas_bc"))
           .agg(F.count("*").alias("casos"),
                F.round(F.avg("target") * 100, 2).alias("bad_rate_pct")))
